# Prosody × Social Response — CPU study

**Targets:** 3 million Reddit comments, 3 million X posts, 3 million Bluesky posts, plus Stanford SHP-2 paired preferences, EPITOME empathy, and Cornell ChangeMyView persuasion.

- CPU-only; batched streaming Parquet and resumable checkpoints.
- Separate engagement, empathy, persuasion and paired preference models.
- Estimates lexical stress with CMUdict; not measured spoken prosody.
- No impression counts: these outputs are NOT CTR.
- Requires explicit source rights approval; no site scraping or API bypass.

Data sources and schemas: https://huggingface.co/datasets/HuggingFaceGECLM/REDDIT_comments ; https://huggingface.co/datasets/blockmesh/tweets ; https://huggingface.co/datasets/Roronotalt/bluesky-five-million ; https://huggingface.co/datasets/alpindale/two-million-bluesky-posts ; https://huggingface.co/datasets/stanfordnlp/SHP-2 ; https://github.com/behavioral-data/Empathy-Mental-Health ; https://convokit.cornell.edu/documentation/winning.html

The current Reddit researcher program has restrictive eligibility. Public availability of a third-party dataset does not by itself establish reuse rights. Verify terms first.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets>=3.0,<5", "huggingface_hub>=0.26", "cmudict>=1.0.31",
    "duckdb>=1.1", "pyarrow>=16", "numpy>=1.26", "pandas>=2.0",
    "requests>=2.31", "scipy>=1.11", "statsmodels>=0.14"])

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import sys
COMMIT = "ec0fc91325fd3d17aaed148fe1fe84751df251e4"
URL = f"https://raw.githubusercontent.com/cheeseonamonkey/Iambic.cc/{COMMIT}/research/prosody/pipeline.py"
urlretrieve(URL, "pipeline.py")
if "pipeline" in sys.modules: del sys.modules["pipeline"]
import pipeline as p
p.ROOT = Path("/kaggle/working/prosody_results" if Path("/kaggle/working").exists()
              else "/content/prosody_results" if Path("/content").exists()
              else "./prosody_results")
p.MODE = "pilot"    # Set "full" after a successful smoke test.
p.APPROVED = {"reddit": False, "x": False, "bluesky": False}
# Set any platform True ONLY after verifying access and processing permissions.
p.BATCH = 5000
p.self_test()
print("Output:", p.ROOT)
print("Sources:", [(s["id"], s["quota"]) for s in p.source_specs()])

## Access attestation

After checking access rights, set the corresponding platform flags to True in the cell above and rerun it. A platform left False is skipped explicitly. Reddit-gated supplemental datasets are not fetched without Reddit permission. No model training needs GPU.

In [ ]:
# Re-run after interruptions to resume completed 5,000-row chunks.
statuses = p.platforms()
statuses

In [ ]:
# Independent labeled studies: paired preference, empathy, actual persuasion.
p.supplements()

In [ ]:
# Cluster-robust analyses + Benjamini-Hochberg FDR correction.
summary = p.analyze()
summary

In [ ]:
from IPython.display import display, Markdown
report_path = p.ROOT / "reports" / "report.md"
if report_path.exists(): display(Markdown(report_path.read_text(encoding="utf8")[:20000]))
print("Files:", list((p.ROOT / "reports").glob("*")))

## Interpretation

1. Report observational associations only: neither causality nor CTR is measured.
2. CMUdict lexical stress is an approximate written-meter feature. Validate against human scansion.
3. Streaming shuffle is not a globally uniform sample; posts may repeat across Bluesky archives.
4. Blind randomized rewriting is required to isolate meter independently of semantics.
5. Keep user text and source-derived rows private unless reuse and redistribution are permitted.
6. Report includes clustered uncertainty estimates, q-values and a held-out SHP-2 comparison subset.

In [ ]:
import shutil
if (p.ROOT / "reports").exists():
    print(shutil.make_archive(str(p.ROOT / "prosody_report"), "zip",
                              root_dir=p.ROOT / "reports"))